# NB3 — Mispronunciation Detection (scoring + benchmark + product verdict)

Given a user recitation and the **expected verse**, this notebook:
1. phonemizes the expected diacritized verse -> canonical phoneme sequence (MSA phonetiser),
2. CTC-decodes the audio with the NB2 model -> predicted phoneme sequence,
3. aligns the two and **flags errors** (substitution / deletion / insertion), showing the correct phoneme,
4. maps each error back to the word it belongs to (for the app UI),
5. (optional) reports F1 / PER on the `QuranMB.v2` benchmark.

### Week-1 reality check (zero training)
Set `MODEL_DIR` to a **pre-trained Iqra'Eval model** (e.g. `IqraEval/Iqra_wav2vec2_base`)
to measure where an off-the-shelf phoneme model lands on YOUR audio before training
anything. Then point it at your NB2 `final/` to compare.

In [1]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa jiwer
# MSA phonetiser (inference-time G2P for arbitrary expected verses):
!git clone -q https://github.com/Iqra-Eval/MSA_phonetiser.git || echo 'already cloned'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 83.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 72.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompa

In [2]:
import os, sys, json, re
import numpy as np
import torch

# ---- PATHS / CONFIG -----------------------------------------------------
# Your trained model from NB2 (final PER 0.071). Attach the model dataset you
# saved and point MODEL_DIR at the folder that contains config.json + the
# safetensors weights. Verify with:  !find /kaggle/input -name config.json
MODEL_DIR = "/kaggle/input/datasets/qasimmaajid/model-3/w2vbert-quran-phoneme-real/final"   # EDIT to your model path

SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PHONETISER_DIR = "MSA_phonetiser"
print("device:", DEVICE, "| model:", MODEL_DIR)
print("model dir exists:", os.path.isdir(MODEL_DIR))

device: cuda | model: /kaggle/input/datasets/qasimmaajid/model-3/w2vbert-quran-phoneme-real/final
model dir exists: True


In [3]:
from transformers import AutoModelForCTC, AutoProcessor, Wav2Vec2BertForCTC, AutoFeatureExtractor, Wav2Vec2CTCTokenizer

# Load model + the pieces needed to decode CTC output to phoneme symbols.
model = AutoModelForCTC.from_pretrained(MODEL_DIR).to(DEVICE).eval()
try:
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    feature_extractor = processor.feature_extractor
    tokenizer = processor.tokenizer
except Exception:
    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
    tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MODEL_DIR)

id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
blank_id = tokenizer.pad_token_id
print("vocab size:", len(id2tok), "| blank id:", blank_id)

2026-06-29 12:52:37.148056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782737557.400629      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782737557.471238      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782737558.045297      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782737558.045344      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782737558.045347      58 computation_placer.cc:177] computation placer alr

vocab size: 73 | blank id: 0


In [4]:
def recognise_phonemes(audio_array):
    """audio (float32 @16k) -> predicted phoneme list via CTC greedy decode.
    Silence tokens are dropped: <sil> is not a pronunciation error, and the
    model emits it between words, which would otherwise misalign vs the
    canonical sequence."""
    feats = feature_extractor(np.asarray(audio_array, dtype=np.float32),
                              sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = {k: v.to(DEVICE) for k, v in feats.items()}
    with torch.no_grad():
        logits = model(**feats).logits[0]
    ids = logits.argmax(-1).tolist()
    out, prev = [], None
    for i in ids:
        if i != prev and i != blank_id:
            sym = id2tok.get(int(i), "<unk>")
            if sym not in ("<sil>", "sil"):
                out.append(sym)
        prev = i
    return out

In [5]:
# ---- Canonical G2P: expected diacritized verse -> phoneme list ----------
sys.path.insert(0, PHONETISER_DIR)
_phonetise = None
try:
    from phonetiser import phonetise_Arabic
    _phonetise = phonetise_Arabic.phonetise
except Exception as e:
    print("phonetiser import failed, will fall back:", e)

def canonical_phonemes(tashkeel_text):
    """Return a flat phoneme list for a diacritized Arabic string.
    Uses the MSA phonetiser if available; output is space-separated symbols
    matching the training vocab convention (shadda=doubled symbol, long vowels aa/ii0/uu0).
    """
    if _phonetise is None:
        raise RuntimeError("MSA phonetiser unavailable; check the clone in cell 1.")
    _, with_spaces, _ = _phonetise(tashkeel_text)
    # with_spaces: list (per utterance) of space-separated phoneme strings
    line = with_spaces[0] if isinstance(with_spaces, (list, tuple)) else with_spaces
    return [t for t in str(line).split() if t]

# Smoke test
try:
    print(canonical_phonemes("بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"))
except Exception as e:
    print("smoke test note:", e)

phoetising utterance
bisomi Alla~hi Alra~Homaٰni Alra~Hiymi
['b', 'i0', 's', 'm', 'i0']
['l', 'l', 'aa', 'h', 'i0']
['l', 'r', 'aa', 'H', 'm', 'a', 'n', 'i0']
['l', 'r', 'aa', 'H', 'ii0', 'm', 'i0']
['b', 'i0', 's', 'm', 'i0', 'l', 'l', 'aa', 'h', 'i0', 'l', 'r', 'aa', 'H', 'm', 'a', 'n', 'i0', 'l', 'r', 'aa', 'H', 'ii0', 'm', 'i0', 'sil']


In [6]:
# ---- Normalise phonetiser output to the model's alphabet -------------------
# The model emits its training-vocab symbols (a, aa, i, ii, ll, rr, SS ...).
# The MSA phonetiser emits extra decoration the model never saw:
#   * subscript context tags:  i0->i, ii0->ii, I0->I, AA0->AA, etc. (strip "0")
#   * a SHADDA as a doubled-with-subscript pair: i0i0 / I0I0 -> long vowel
#   * word-initial hamza "<" and the "al-" article that ELIDE in continuous
#     recitation (the model correctly does NOT pronounce them) -> drop "<"
#   * "sil" silence -> drop
# We build the map programmatically so every subscript/case variant is covered.
model_vocab = set(id2tok.values()) - {"<pad>", "<unk>", "<s>", "</s>", "|", "<sil>"}

import re as _re
def _norm_one(p):
    if p in ("sil", "<sil>"):        # silence
        return None
    if p == "<":                      # elided word-initial hamza
        return None
    # collapse a doubled-with-subscript vowel (shadda on a vowel) e.g. i0i0 -> ii
    m = _re.fullmatch(r"([A-Za-z]+)0([A-Za-z]+)0", p)
    if m and m.group(1) == m.group(2):
        base = m.group(1)
        return base if len(base) > 1 else base + base   # i->ii, ii->ii
    # strip a trailing subscript digit: i0->i, ii0->ii, I1->I, AA0->AA
    m = _re.fullmatch(r"([A-Za-z]+)[0-9]", p)
    if m:
        return m.group(1)
    return p

def normalise_phonemes(seq):
    out = []
    for p in seq:
        q = _norm_one(p)
        if q:
            out.append(q)
    return out

# Diagnostic: phonemize a Fatiha chunk, normalise, report leftover unknowns.
probe = canonical_phonemes(
    "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ"
)
probe_syms = set(normalise_phonemes(probe))
unknown = sorted(probe_syms - model_vocab)
print("model vocab size:", len(model_vocab))
print("canonical symbols after normalise:", sorted(probe_syms))
print("still-unknown symbols:", unknown)
print("OK\n" if not unknown else f">>> still unmapped: {unknown}\n")

phoetising utterance
bisomi Alla~hi Alra~Homaٰni Alra~Hiymi AloHamodu lila~hi rabi~ AloEaAlamiyna
['b', 'i0', 's', 'm', 'i0']
['l', 'l', 'aa', 'h', 'i0']
['l', 'r', 'aa', 'H', 'm', 'a', 'n', 'i0']
['l', 'r', 'aa', 'H', 'ii0', 'm', 'i0']
['l', 'H', 'a', 'm', 'd', 'u0']
['l', 'i0', 'l', 'aa', 'h', 'i0']
['r', 'a', 'b', 'i0i0']
Unacceptable char when finding stress syllable: i0i0 cvc

['l', 'E', 'aa', 'l', 'a', 'm', 'ii0', 'n', 'a']
model vocab size: 68
canonical symbols after normalise: ['E', 'H', 'a', 'aa', 'b', 'd', 'h', 'i', 'ii', 'l', 'm', 'n', 'r', 's', 'u']
still-unknown symbols: []
OK



In [7]:
# ---- Alignment + error tagging (Levenshtein over phoneme symbols) -------
def align(ref, hyp):
    """Return list of ops: ('ok'|'sub'|'del'|'ins', ref_sym, hyp_sym).
    ref = canonical (expected), hyp = recognised (what was said).
    'del' = expected phoneme missing in speech (e.g. dropped shadda half).
    'sub' = wrong phoneme said (e.g. fatha instead of damma)."""
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    i, j, ops = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + (0 if ref[i-1]==hyp[j-1] else 1):
            ops.append(("ok" if ref[i-1]==hyp[j-1] else "sub", ref[i-1], hyp[j-1])); i-=1; j-=1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            ops.append(("del", ref[i-1], None)); i-=1
        else:
            ops.append(("ins", None, hyp[j-1])); j-=1
    return ops[::-1]

In [8]:
# ---- Per-word verdict, CONNECTED-SPEECH AWARE -------------------------------
# Goal: flag ONLY genuine mispronunciations. The strict per-word reference (from
# the isolated-word phonetiser) disagrees with correct CONNECTED recitation in
# three predictable, NON-error ways. We forgive exactly those, and nothing else:
#
#  1) ELISION: word-initial hamzat-wasl '<' and the 'al-' article elide
#     mid-utterance. The model correctly omits them -> strip from canonical.
#  2) SUN-LETTER ASSIMILATION (idgham): 'al-' + sun letter doubles the next
#     consonant (al-rahman -> "rr"). The model hears the doubled form correctly,
#     the isolated reference has the single form -> collapse gemination on BOTH
#     sides so the assimilation is never a 'sub'.
#  3) POST-ASSIMILATION VOWEL REDUCTION: right after that assimilated consonant
#     the long vowel surfaces short (al-laah -> "ll a h", not "ll aa h"). That
#     specific aa<->a difference is connected-speech-correct, so we forgive a
#     vowel-LENGTH-only substitution *at that position only*.
#
# CRUCIAL: a vowel-length change ANYWHERE ELSE is a real madd (elongation) error
# and STILL flags. Every consonant substitution/deletion STILL flags. We are not
# blanket-ignoring vowels — only the three connected-speech artifacts above.
MIN_ERRORS_PER_WORD = 1

# consonants that can carry shadda from assimilation/gemination
_DOUBLED = {"ll","rr","nn","ss","SS","dd","DD","tt","TT","bb","mm","kk",
            "ff","qq","jj","zz","ZZ","hh","HH","xx","gg","**","cc","yy","ww"}

# vowels whose ONLY difference is length (short <-> long); same colour.
_VOWEL_LEN_PAIRS = {("a","aa"),("aa","a"),("i","ii"),("ii","i"),
                    ("u","uu"),("uu","u"),
                    ("A","AA"),("AA","A"),("I","II"),("II","I"),
                    ("U","UU"),("UU","U")}

def _is_doubled(p):
    return p in _DOUBLED

def _collapse_gemination(seq):
    """Map a doubled consonant token (ll, rr, SS ...) to its single form so that
    sun-letter assimilation / shadda is not counted as a substitution in the
    per-word display. Long vowels (aa, ii, uu) are NOT touched."""
    return [p[0] if p in _DOUBLED else p for p in seq]

def word_phonemes(w):
    """Canonical phonemes for one word, normalised to the model vocab."""
    return normalise_phonemes(canonical_phonemes(w))

def word_phonemes_connected(w):
    """Same, but with connected-speech elisions removed so it matches fluent
    mid-utterance recitation: drop leading hamzat-wasl '<' and the 'al-' lam."""
    ph = list(word_phonemes(w))
    if ph and ph[0] == "<":          # hamzat-wasl elides
        ph = ph[1:]
    if len(ph) >= 2 and ph[0] == "a" and ph[1] == "l":   # 'al-' article
        ph = ph[2:]
    return ph

def analyse_recitation(audio_array, expected_tashkeel):
    """Per-word pass/flag. Flags only genuine mispronunciations: forgives the
    three connected-speech artifacts (elision, assimilation, post-assimilation
    vowel reduction); keeps all consonant errors and all other madd changes."""
    words = expected_tashkeel.split()
    ref_flat, ref_word_of, ref_doubled = [], [], []
    for wi, w in enumerate(words):
        ph = word_phonemes_connected(w)
        ref_flat += ph
        ref_word_of += [wi] * len(ph)
        ref_doubled += [_is_doubled(p) for p in ph]   # was this consonant geminated?
    hyp = recognise_phonemes(audio_array)

    # collapse gemination on BOTH sides so assimilation itself is never a 'sub'
    ref_cmp = _collapse_gemination(ref_flat)
    hyp_cmp = _collapse_gemination(hyp)
    ops = align(ref_cmp, hyp_cmp)

    errors = {wi: [] for wi in range(len(words))}
    rp = 0
    prev_ref_was_doubled = False   # did the previous canonical phoneme assimilate?
    for op, r, h in ops:
        if op in ("ok", "sub", "del"):
            wi = ref_word_of[rp] if rp < len(ref_word_of) else len(words) - 1
            orig_ref = ref_flat[rp] if rp < len(ref_flat) else r
            if op == "sub":
                vowel_len_only = (orig_ref, h) in _VOWEL_LEN_PAIRS
                # forgive vowel-length ONLY right after an assimilated consonant
                forgive = vowel_len_only and prev_ref_was_doubled
                if not forgive:
                    errors[wi].append((op, orig_ref, h))
            elif op == "del":
                errors[wi].append((op, orig_ref, h))
            # update context: was THIS canonical phoneme a geminated consonant?
            prev_ref_was_doubled = ref_doubled[rp] if rp < len(ref_doubled) else False
            rp += 1
        # pure insertions (op == 'ins') are NOT counted as flags

    report = []
    for wi, w in enumerate(words):
        errs = errors[wi]
        report.append({
            "word": w,
            "verdict": "flag" if len(errs) >= MIN_ERRORS_PER_WORD else "pass",
            "errors": errs,
            "correct_phonemes": word_phonemes_connected(w),
        })
    return report

In [9]:
# ---- Demo on one clip ---------------------------------------------------
import soundfile as sf, subprocess
import jiwer

def load_audio(path):
    if path.endswith((".opus", ".mp3", ".m4a")):
        cmd = ["ffmpeg","-loglevel","error","-i",path,"-f","f32le","-ar",str(SAMPLING_RATE),"-ac","1","pipe:1"]
        raw = subprocess.run(cmd, capture_output=True).stdout
        return np.frombuffer(raw, dtype=np.float32).copy()
    a, sr = sf.read(path, dtype="float32")
    return a

DEMO_AUDIO = "/kaggle/input/datasets/qasimmaajid/fatiha-clip/audio.opus"   # EDIT
DEMO_VERSE = "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ الرَّحْمَٰنِ الرَّحِيمِ مَالِكِ يَوْمِ الدِّينِ إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ اهْدِنَا الصِّرَاطَ الْمُسْتَقِيمَ صِرَاطَ الَّذِينَ أَنْعَمْتَ عَلَيْهِمْ غَيْرِ الْمَغْضُوبِ عَلَيْهِمْ وَلَا الضَّالِّينَ"

if os.path.exists(DEMO_AUDIO):
    audio = load_audio(DEMO_AUDIO)
    print(f"audio: {len(audio)/SAMPLING_RATE:.1f}s\n")

    heard    = recognise_phonemes(audio)
    expected = normalise_phonemes(canonical_phonemes(DEMO_VERSE))
    print("EXPECTED phonemes :", " ".join(expected))
    print("MODEL HEARD       :", " ".join(heard))

    # ---- PRIMARY METRIC: phoneme error rate over the whole utterance --------
    # This is the trustworthy number. Per-word flags below are a secondary view
    # and can over-flag because the phonetiser inserts word-initial hamza/"al-"
    # that elide in continuous recitation (tajweed) — that's a reference quirk,
    # not a model error. The overall PER is robust to those boundary effects.
    per = jiwer.wer(" ".join(expected), " ".join(heard))
    print(f"\n>>> Phoneme accuracy on this clip: {1-per:.1%}  (PER {per:.3f})")
    print(f"    Old whisper-medium on the same clip: ~31% correct-word-won.\n")

    # ---- Secondary: per-word flags (for the app UI) -------------------------
    rep = analyse_recitation(audio, DEMO_VERSE)
    n_pass = sum(r["verdict"] == "pass" for r in rep)
    for r in rep:
        tag = "FLAG" if r["verdict"]=="flag" else "ok  "
        errs = "; ".join(f"{op}:{rr}->{hh}" for op,rr,hh in r["errors"]) or "-"
        print(f"[{tag}] {r['word']:15s} correct={' '.join(r['correct_phonemes'])}  errors={errs}")
    print(f"\nper-word (strict): {n_pass}/{len(rep)} passed ({n_pass/len(rep):.0%}) "
          f"— strict word boundaries over-flag; trust the PER above.")
else:
    print("Set DEMO_AUDIO to a real clip to run the demo.")

audio: 36.2s

phoetising utterance
bisomi Alla~hi Alra~Homaٰni Alra~Hiymi AloHamodu lila~hi rabi~ AloEaAlamiyna Alra~Homaٰni Alra~Hiymi maAliki yawomi Aldi~yni <iya~Aka naEobudu wa<iya~Aka nasotaEiynu AhodinaA AlSi~raATa Alomusotaqiyma SiraATa Ala~*iyna >anoEamota Ealayohimo gayori AlomagoDuwbi Ealayohimo walaA AlDa~Ali~yna
['b', 'i0', 's', 'm', 'i0']
['l', 'l', 'aa', 'h', 'i0']
['l', 'r', 'aa', 'H', 'm', 'a', 'n', 'i0']
['l', 'r', 'aa', 'H', 'ii0', 'm', 'i0']
['l', 'H', 'a', 'm', 'd', 'u0']
['l', 'i0', 'l', 'aa', 'h', 'i0']
['r', 'a', 'b', 'i0i0']
Unacceptable char when finding stress syllable: i0i0 cvc

['l', 'E', 'aa', 'l', 'a', 'm', 'ii0', 'n', 'a']
['l', 'r', 'aa', 'H', 'm', 'a', 'n', 'i0']
['l', 'r', 'aa', 'H', 'ii0', 'm', 'i0']
['m', 'aa', 'l', 'i0', 'k', 'i0']
['y', 'a', 'w', 'm', 'i0']
['l', 'd', 'i0i0', 'ii0', 'n', 'i0']
Unacceptable char when finding stress syllable: i0i0 cc

['<', 'i0', 'y', 'aa', 'aa', 'k', 'a']
['n', 'a', 'E', 'b', 'u0', 'd', 'u0']
['w', 'a', '<', 'i0', '

In [10]:
# ---- Benchmark on QuranMB.v2 (PER, and detection F1 if GT phonemes available) ----
# QuranMB.v2 public rows expose ID + audio. The verbatim-annotation phoneme
# references live in the gated companion sets (IqraEval_Test_GT / test_references);
# accept their terms on HF and set HF_TOKEN to compute detection F1.
# Without them, we still report PER vs the CANONICAL phonemes as a sanity metric.
from datasets import load_dataset
import jiwer

RUN_BENCHMARK = False   # set True once you have the test set + (optional) GT
if RUN_BENCHMARK:
    qmb = load_dataset("IqraEval/QuranMB.v2", split="test")
    print(qmb)
    pers = []
    for ex in qmb:
        hyp = recognise_phonemes(ex["audio"]["array"])
        # If a canonical/tashkeel field exists, phonemize it; else skip PER.
        ref_text = ex.get("tashkeel_sentence") or ex.get("sentence")
        if ref_text:
            ref = canonical_phonemes(ref_text)
            pers.append(jiwer.wer(" ".join(ref), " ".join(hyp)))
    if pers:
        print(f"PER vs canonical: {np.mean(pers):.4f} over {len(pers)} clips")
    print("For detection F1 (TA/FR/FA/CD), align hyp against BOTH the canonical and "
          "the annotated-verbatim phoneme sequences from the GT set.")

### Notes
- **Precision tuning** (`MIN_ERRORS_PER_WORD`) is your main product dial: raise it so the
  app stays quiet unless confident — important for a Quran app where false flags are costly.
- **Detection F1 on QuranMB** needs the gated GT phoneme annotations; accept the dataset
  terms on HuggingFace and re-run with `HF_TOKEN` set.
- The phonetiser output convention must match the training vocab; if symbols differ
  (e.g. subscripts), add a small mapping table here mirroring AraS2P Table 2.